In [1]:
!pip install -U torch
!pip install -U scikit-learn
!pip install -U hyperopt
# !pip install -U numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 88.5 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlin

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from pathlib import Path

DATA_FOLDER = Path("/content/drive/MyDrive/DSCI599/data")

assert DATA_FOLDER.exists()

# Purpose of Notebook

Optimize sillhouette score of clustering by tweaking the parameters of the data
Use bayesian optimization because it minimizes search space

I did this previosuly but the notebook dissapeared lol

In [4]:
import torch
import numpy as np

embeddings_path = DATA_FOLDER / "source_ip_embeddings_3.pt"
assert embeddings_path.exists()

# with open(embeddings_path, 'r')
emb = torch.load(embeddings_path, map_location=torch.device("cpu"))
emb.requires_grad = False
emb = emb.numpy()

emb.shape

(132956, 64)

# sampling

to avoid hdbscan the whole dataset for optimizing, just do it on a small sample
this approximates the silhouette score

In [8]:
from sklearn.utils import resample
NUM_SAMPLES = 5000
SEED = 42

emb_sample = resample(emb, n_samples=NUM_SAMPLES, random_state=SEED)
emb_sample.shape

(5000, 64)

In [9]:
from sklearn.cluster import HDBSCAN
from sklearn.metrics import silhouette_score
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK
import numpy as np

search_space = {
    'min_cluster_size': hp.quniform('min_cluster_size', 5, 50, 1),
    'min_samples': hp.quniform('min_samples', 1, 30, 1),
    'cluster_selection_epsilon': hp.uniform('cluster_selection_epsilon', 0.0, 1.0),
    'alpha': hp.uniform('alpha', 0.1, 2.0),
    'cluster_selection_method': hp.choice('cluster_selection_method', ['eom', 'leaf'])
}

def objective(params):
  # make sure it's int
  params['min_cluster_size'] = int(params['min_cluster_size'])
  params['min_samples'] = int(params['min_samples'])

  # to speed things up
  # params['approx_min_span_tree'] = True
  params['n_jobs'] = -1
  try:
    hdb = HDBSCAN(**params)
    labels = hdb.fit_predict(emb_sample)

    # if all points are noise or only have one cluster -> silhouette won't work
    n_clusters = len(set(labels))
    if -1 in labels:
      n_clusters =- 1
    if n_clusters <= 1:
      loss = 1.0

    score = silhouette_score(emb_sample, labels)
    loss = -score

  # if fails, means the params are bad
  except Exception as e:
    loss = 1.0

  return {"loss": loss, 'status': STATUS_OK}

In [10]:
trials = Trials()

best = fmin(
    fn=objective,
    space=search_space,
    algo=tpe.suggest,
    max_evals=50,
    trials=trials,
    rstate=np.random.default_rng(42)
)

100%|██████████| 50/50 [03:45<00:00,  4.52s/trial, best loss: -0.237785205245018]


In [11]:
print("best params", best)

best params {'alpha': np.float64(0.25256825415766593), 'cluster_selection_epsilon': np.float64(0.07245511032486793), 'cluster_selection_method': np.int64(0), 'min_cluster_size': np.float64(5.0), 'min_samples': np.float64(20.0)}


# try for KMeans

In [12]:
import numpy as np
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK

km_search_space = {
    'n_clusters': hp.quniform('n_clusters', 2, 10, 1),
    'init': hp.choice('init', ['k-means++', 'random']),
    'n_init': hp.quniform('n_init', 5, 20, 1),          # how many times to run KMeans
    'max_iter': hp.quniform('max_iter', 100, 500, 50),
}

def km_objective(params):
  # make sure it's int
  params['n_clusters'] = int(params['n_clusters'])
  params['n_init'] = int(params['n_init'])
  params['max_iter'] = int(params['max_iter'])


  # to speed things up
  # params['n_jobs'] = -1
  try:
    km = KMeans(**params)
    labels = km.fit_predict(emb_sample)

    # if all points are noise or only have one cluster -> silhouette won't work
    n_clusters = len(set(labels))
    if -1 in labels:
      n_clusters =- 1
    if n_clusters <= 1:
      loss = 1.0

    score = silhouette_score(emb_sample, labels)
    loss = -score

  # if fails, means the params are bad
  except Exception as e:
    print(e)
    loss = 1.0

  return {"loss": loss, 'status': STATUS_OK}

In [13]:
km_trials =Trials()
km_best = fmin(
    fn=km_objective,
    space=km_search_space,
    algo=tpe.suggest,
    max_evals=50,
    trials=km_trials,
    rstate=np.random.default_rng(42)
)

print("best kmeans", km_best)

100%|██████████| 50/50 [00:46<00:00,  1.07trial/s, best loss: -0.2637980580329895]
best kmeans {'init': np.int64(0), 'max_iter': np.float64(450.0), 'n_clusters': np.float64(7.0), 'n_init': np.float64(17.0)}


# repeat for spectral

In [14]:
from sklearn.cluster import SpectralClustering
from sklearn.metrics import silhouette_score
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK
import numpy as np

sc_search_space = {
    'n_clusters': hp.quniform('n_clusters', 2, 10, 1),
    'affinity': hp.choice('affinity', ['nearest_neighbors', 'rbf']),
    'gamma': hp.uniform('gamma', 0.1, 2.0),
    'n_neighbors': hp.quniform('n_neighbors', 5, 20, 1)
}

def sc_objective(params):
    n_clusters = int(params['n_clusters'])
    affinity = params['affinity']
    n_neighbors = int(params['n_neighbors'])
    gamma = params['gamma']

    try:
        clustering_args = {
            'n_clusters': n_clusters,
            'assign_labels': 'kmeans',
            'random_state': 42,
            'affinity': affinity
        }

        if affinity == 'nearest_neighbors':
            clustering_args['n_neighbors'] = n_neighbors
        elif affinity == 'rbf':
            clustering_args['gamma'] = gamma

        model = SpectralClustering(**clustering_args)
        labels = model.fit_predict(emb_sample)

        if len(set(labels)) <= 1:
            return {'loss': 1.0, 'status': STATUS_OK}

        score = silhouette_score(emb_sample, labels)
        return {'loss': -score, 'status': STATUS_OK}

    except Exception as e:
        print(e)
        return {'loss': 1.0, 'status': STATUS_OK}

In [15]:
sc_trials =Trials()
sc_best = fmin(
    fn=sc_objective,
    space=sc_search_space,
    algo=tpe.suggest,
    max_evals=50,
    trials=sc_trials,
    rstate=np.random.default_rng(42)
)

print("best sc", sc_best)

  2%|▏         | 1/50 [00:46<38:19, 46.92s/trial, best loss: -0.25063657760620117]


KeyboardInterrupt: 

In [16]:
print("best sc", sc_best)

NameError: name 'sc_best' is not defined

# saving clusters

In [17]:
km_best['init'] = 'random'
km_best['max_iter'] = int(km_best['max_iter'])
km_best['n_clusters'] = int(km_best['n_clusters'])
km_best['n_init'] = int(km_best['n_init'])


In [19]:
hdb = HDBSCAN(**best)
km = KMeans(**km_best)
# sc = SpectralClustering(**sc_best)

In [20]:
labels = km.fit_predict(emb)

In [21]:
np.unique(labels, return_counts=True)

(array([0, 1, 2, 3, 4, 5, 6], dtype=int32),
 array([12897,  3600,  8582, 19618,  2232, 83032,  2995]))

In [22]:
np.save(DATA_FOLDER / "km_newlabels_2", labels)